# **With torch**

In [28]:
import numpy as np
import torch
import torchvision.transforms as T
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

# Load data
data_input = loadmat('/content/drive/MyDrive/input_a2.mat')['input']
labels = loadmat('/content/drive/MyDrive/label.mat')['label']

input_tensor = torch.from_numpy(data_input).float()
label_tensor = torch.squeeze(torch.from_numpy(labels).long()) - 1   # labels from [1, 6] to [0, 5]
custom_dataset = TensorDataset(input_tensor, label_tensor)

# Split dataset
train_size = int(0.7 * len(custom_dataset))
val_size = int(0.1 * len(custom_dataset))
test_size = len(custom_dataset) - train_size - val_size
train_set, val_set, test_set = torch.utils.data.random_split(custom_dataset, [train_size, val_size, test_size])

# DataLoader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

class FuzzyLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(FuzzyLayer, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.fuzzy_degree = nn.Parameter(torch.Tensor(self.input_dim, self.output_dim))
        self.sigma = nn.Parameter(torch.Tensor(self.input_dim, self.output_dim))

        nn.init.xavier_uniform_(self.fuzzy_degree)  # fuzzy degree init
        nn.init.ones_(self.sigma)  # sigma init

    def forward(self, input_data):
        fuzzy_out = []
        for variable in input_data:
            fuzzy_out_i = torch.exp(-torch.sum(torch.sqrt((variable - self.fuzzy_degree) / (self.sigma ** 2))))
            if torch.isnan(fuzzy_out_i):
                fuzzy_out.append(variable)
            else:
                fuzzy_out.append(fuzzy_out_i)
        return torch.tensor(fuzzy_out, dtype=torch.float)


class HFDNN(nn.Module):
    def __init__(self, input_vector_size, fuzz_vector_size, num_classes, fuzzy_layer_input_dim=1,
                 fuzzy_layer_output_dim=1,
                 dropout_rate=0.5):
        super(HFDNN, self).__init__()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.input_vector_size = input_vector_size
        self.fuzz_vector_size = fuzz_vector_size
        self.num_classes = num_classes
        self.fuzzy_layer_input_dim = fuzzy_layer_input_dim
        self.fuzzy_layer_output_dim = fuzzy_layer_output_dim
        self.dropout_rate = dropout_rate
        self.fuzz_init_linear_layer = nn.Linear(self.input_vector_size, self.fuzz_vector_size)

        fuzzy_rule_layers = []
        for i in range(self.fuzz_vector_size):
            fuzzy_rule_layers.append(FuzzyLayer(fuzzy_layer_input_dim, fuzzy_layer_output_dim))
        self.fuzzy_rule_layers = nn.ModuleList(fuzzy_rule_layers)
        self.dl_linear_1 = nn.Linear(self.input_vector_size, self.input_vector_size)
        self.dl_linear_2 = nn.Linear(self.input_vector_size, self.input_vector_size)
        self.dropout_layer = nn.Dropout(self.dropout_rate)
        self.fusion_layer = nn.Linear(self.input_vector_size * 2, self.input_vector_size)
        self.output_layer = nn.Linear(self.input_vector_size, self.num_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_data):
        fuzz_input = self.fuzz_init_linear_layer(input_data)
        fuzz_output = torch.zeros(input_data.size(), dtype=torch.float, device=self.device)
        for col_idx in range(fuzz_input.size()[1]):
            col_vector = fuzz_input[:, col_idx:col_idx + 1]
            fuzz_col_vector = self.fuzzy_rule_layers[col_idx](col_vector).unsqueeze(0).view(-1, 1)
            fuzz_output[:, col_idx:col_idx + 1] = fuzz_col_vector

        dl_layer_1_output = torch.sigmoid(self.dl_linear_1(input_data))
        dl_layer_2_output = torch.sigmoid(self.dl_linear_2(dl_layer_1_output))
        dl_layer_2_output = self.dropout_layer(dl_layer_2_output)

        cat_fuzz_dl_output = torch.cat([fuzz_output, dl_layer_2_output], dim=1)

        fused_output = torch.sigmoid(self.fusion_layer(cat_fuzz_dl_output))
        fused_output = torch.relu(fused_output)
        output_data = self.log_softmax(self.output_layer(fused_output))
        return output_data

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_epochs = 20
input_dim_size = 72
fuzz_dim_size = 100
num_classes = 6
batch_size_val = 8
learning_rate_val = 10e-3

model = HFDNN(input_dim_size, fuzz_dim_size, num_classes).to(device)

loss_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate_val)

for epoch in range(num_epochs):
    training_loss_val = 0.0
    model.train()
    for data_input_val, labels_val in train_loader:
        # Training steps
        data_input_val, labels_val = data_input_val.to(device), labels_val.to(device)
        optimizer.zero_grad()
        predictions = model(data_input_val)
        loss_val = loss_criterion(predictions, labels_val)
        loss_val.backward()
        optimizer.step()
        training_loss_val += loss_val.item()

    validation_loss_val = 0.0
    model.eval()
    with torch.no_grad():
        for data_val, labels_val in val_loader:
            # Validation steps
            data_val, labels_val = data_val.to(device), labels_val.to(device)
            predictions_val = model(data_val)
            loss_val = loss_criterion(predictions_val, labels_val)
            validation_loss_val += loss_val.item()

    print('Epoch: {:d} - training loss: {:.6f} - validation loss: {:.6f}'.format(epoch, training_loss_val, validation_loss_val))

# Test the model after training using the test_loader
test_loss_val = 0.0
correct_val = 0
total_val = 0

model.eval()  # Set model to evaluation mode
with torch.no_grad():
    for data_test, labels_test in test_loader:
        data_test, labels_test = data_test.to(device), labels_test.to(device)
        predictions_test = model(data_test)
        loss_test = loss_criterion(predictions_test, labels_test)
        test_loss_val += loss_test.item()
        _, predicted_val = torch.max(predictions_test, 1)
        total_val += labels_test.size(0)
        correct_val += (predicted_val == labels_test).sum().item()

accuracy_val = correct_val / total_val
print('Test Loss: {:.6f} - Test Accuracy: {:.2f}%'.format(test_loss_val, accuracy_val * 100))

Epoch: 0 - training loss: 900.773282 - validation loss: 128.149532
Epoch: 1 - training loss: 879.199701 - validation loss: 126.874240
Epoch: 2 - training loss: 865.887683 - validation loss: 123.060866
Epoch: 3 - training loss: 851.103351 - validation loss: 121.556253
Epoch: 4 - training loss: 840.518619 - validation loss: 121.776117
Epoch: 5 - training loss: 826.431894 - validation loss: 119.470274
Epoch: 6 - training loss: 819.580667 - validation loss: 118.342520
Epoch: 7 - training loss: 807.065879 - validation loss: 116.417236
Epoch: 8 - training loss: 798.036986 - validation loss: 116.949902
Epoch: 9 - training loss: 791.322120 - validation loss: 116.060992
Epoch: 10 - training loss: 783.049027 - validation loss: 114.864560
Epoch: 11 - training loss: 776.429223 - validation loss: 113.354675
Epoch: 12 - training loss: 767.764474 - validation loss: 112.666911
Epoch: 13 - training loss: 764.285795 - validation loss: 112.967847
Epoch: 14 - training loss: 758.526634 - validation loss: 1

In [121]:
import numpy as np
import torch
import torchvision.transforms as T
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

# Load data
data_input = loadmat('/content/drive/MyDrive/input_a2.mat')['input']
labels = loadmat('/content/drive/MyDrive/label.mat')['label']

input_tensor = torch.from_numpy(data_input).float()
label_tensor = torch.squeeze(torch.from_numpy(labels).long()) - 1   # labels from [1, 6] to [0, 5]
custom_dataset = TensorDataset(input_tensor, label_tensor)

# Split dataset
train_size = int(0.7 * len(custom_dataset))
val_size = int(0.1 * len(custom_dataset))
test_size = len(custom_dataset) - train_size - val_size
train_set, val_set, test_set = torch.utils.data.random_split(custom_dataset, [train_size, val_size, test_size])

# DataLoader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

class FuzzyLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(FuzzyLayer, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.fuzzy_degree = nn.Parameter(torch.Tensor(self.input_dim, self.output_dim))
        self.sigma = nn.Parameter(torch.Tensor(self.input_dim, self.output_dim))

        nn.init.xavier_uniform_(self.fuzzy_degree)  # fuzzy degree init
        nn.init.ones_(self.sigma)  # sigma init

    def forward(self, input_data):
        fuzzy_out = []
        for variable in input_data:
            fuzzy_out_i = torch.exp(-torch.sum(torch.sqrt((variable - self.fuzzy_degree) / (self.sigma ** 2))))
            if torch.isnan(fuzzy_out_i):
                fuzzy_out.append(variable)
            else:
                fuzzy_out.append(fuzzy_out_i)
        return torch.tensor(fuzzy_out, dtype=torch.float)


class HFDNN(nn.Module):
    def __init__(self, input_vector_size, fuzz_vector_size, num_classes, fuzzy_layer_input_dim=1,
                 fuzzy_layer_output_dim=1,
                 dropout_rate=0.5):
        super(HFDNN, self).__init__()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.input_vector_size = input_vector_size
        self.fuzz_vector_size = fuzz_vector_size
        self.num_classes = num_classes
        self.fuzzy_layer_input_dim = fuzzy_layer_input_dim
        self.fuzzy_layer_output_dim = fuzzy_layer_output_dim
        self.dropout_rate = dropout_rate
        self.fuzz_init_linear_layer = nn.Linear(self.input_vector_size, self.fuzz_vector_size)

        fuzzy_rule_layers = []
        for i in range(self.fuzz_vector_size):
            fuzzy_rule_layers.append(FuzzyLayer(fuzzy_layer_input_dim, fuzzy_layer_output_dim))
        self.fuzzy_rule_layers = nn.ModuleList(fuzzy_rule_layers)
        self.dl_linear_1 = nn.Linear(self.input_vector_size, self.input_vector_size)
        self.dl_linear_2 = nn.Linear(self.input_vector_size, self.input_vector_size)
        self.dropout_layer = nn.Dropout(self.dropout_rate)
        self.fusion_layer = nn.Linear(self.input_vector_size * 2, self.input_vector_size)
        self.output_layer = nn.Linear(self.input_vector_size, self.num_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_data):
        fuzz_input = self.fuzz_init_linear_layer(input_data)
        fuzz_output = torch.zeros(input_data.size(), dtype=torch.float, device=self.device)
        for col_idx in range(fuzz_input.size()[1]):
            col_vector = fuzz_input[:, col_idx:col_idx + 1]
            fuzz_col_vector = self.fuzzy_rule_layers[col_idx](col_vector).unsqueeze(0).view(-1, 1)
            fuzz_output[:, col_idx:col_idx + 1] = fuzz_col_vector

        dl_layer_1_output = torch.sigmoid(self.dl_linear_1(input_data))
        dl_layer_2_output = torch.sigmoid(self.dl_linear_2(dl_layer_1_output))
        dl_layer_2_output = self.dropout_layer(dl_layer_2_output)

        cat_fuzz_dl_output = torch.cat([fuzz_output, dl_layer_2_output], dim=1)

        fused_output = torch.sigmoid(self.fusion_layer(cat_fuzz_dl_output))
        fused_output = torch.relu(fused_output)
        output_data = self.log_softmax(self.output_layer(fused_output))
        return output_data

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_epochs = 20
input_dim_size = 72
fuzz_dim_size = 90
num_classes = 6
batch_size_val = 8
learning_rate_val = 0.01

model = HFDNN(input_dim_size, fuzz_dim_size, num_classes).to(device)

loss_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate_val)

for epoch in range(num_epochs):
    training_loss_val = 0.0
    model.train()
    for data_input_val, labels_val in train_loader:
        # Training steps
        data_input_val, labels_val = data_input_val.to(device), labels_val.to(device)
        optimizer.zero_grad()
        predictions = model(data_input_val)
        loss_val = loss_criterion(predictions, labels_val)
        loss_val.backward()
        optimizer.step()
        training_loss_val += loss_val.item()

    validation_loss_val = 0.0
    model.eval()
    with torch.no_grad():
        for data_val, labels_val in val_loader:
            # Validation steps
            data_val, labels_val = data_val.to(device), labels_val.to(device)
            predictions_val = model(data_val)
            loss_val = loss_criterion(predictions_val, labels_val)
            validation_loss_val += loss_val.item()

    print('Epoch: {:d} - training loss: {:.6f} - validation loss: {:.6f}'.format(epoch, training_loss_val, validation_loss_val))

# Test the model after training using the test_loader
test_loss_val = 0.0
correct_val = 0
total_val = 0

model.eval()  # Set model to evaluation mode
with torch.no_grad():
    for data_test, labels_test in test_loader:
        data_test, labels_test = data_test.to(device), labels_test.to(device)
        predictions_test = model(data_test)
        loss_test = loss_criterion(predictions_test, labels_test)
        test_loss_val += loss_test.item()
        _, predicted_val = torch.max(predictions_test, 1)
        total_val += labels_test.size(0)
        correct_val += (predicted_val == labels_test).sum().item()

accuracy_val = correct_val / total_val
print('Test Loss: {:.6f} - Test Accuracy: {:.2f}%'.format(test_loss_val, accuracy_val * 100))

Epoch: 0 - training loss: 28.464723 - validation loss: 5.292593
Epoch: 1 - training loss: 28.049823 - validation loss: 5.230979
Epoch: 2 - training loss: 27.856147 - validation loss: 5.176472
Epoch: 3 - training loss: 27.705699 - validation loss: 5.174725
Epoch: 4 - training loss: 27.715871 - validation loss: 5.148146
Epoch: 5 - training loss: 27.552533 - validation loss: 5.124187
Epoch: 6 - training loss: 27.431483 - validation loss: 5.096641
Epoch: 7 - training loss: 27.282633 - validation loss: 5.082231
Epoch: 8 - training loss: 27.180871 - validation loss: 5.096631
Epoch: 9 - training loss: 27.137659 - validation loss: 5.067231
Epoch: 10 - training loss: 26.996094 - validation loss: 5.043627
Epoch: 11 - training loss: 26.875442 - validation loss: 5.022469
Epoch: 12 - training loss: 26.781426 - validation loss: 5.003536
Epoch: 13 - training loss: 26.652236 - validation loss: 4.989163
Epoch: 14 - training loss: 26.568728 - validation loss: 4.980548
Epoch: 15 - training loss: 26.45400